# Setup
**Make sure to read the instructions carefully!**

If you have other resources used in the Blender project and chose to *make all paths relative*, pack all of them into a zip archive. Alternatively, you can *pack all external file*.

* `blender_version` : Version of blender used to render the scene. You may define your own blender version.
* `blend_file_path` : Path to the blend file after unpacking the zip archive. If blend file is used, this is automatically ignored.
___
* `upload_type` : Select the type of upload method. `gdrive_relative` pulls everything from the folder specified.
* `drive_path` : Path to your blend/zip file relative to the root of your Google Drive if `google_drive` is selected. Must  state the file and its extension (.zip/.blend) **unless** `gdrive_relative` is selected.
* `url_blend` : Specify the URL to the blend/zip file if `url` is selected.
___
* `animation` : Specify whether animation or still image is rendered. If **still image** is used, put the frame number in `start_frame`.
* `start_frame, end_frame` : Specify the start and end frame for animation. You may put same value such as zero for both input to set the default frame in the blend file.
___
* `download_type` : Select the type of download method. `gdrive_direct` enables the frames to be outputted directly to Google Drive (zipping will be disabled).
* `output_name` : Name of the output frames, **do NOT include .blend!** (## for frame number)
* `zip_files` : Archive multiple animation frames automatically into a zip file.
* `drive_output_path` : Path to your frames/zip file in Google Drive.
___
* `gpu_enabled, cpu_enabled` : Toggle GPU and CPU for rendering. CPU might give a slight boost in rendering time but may varies depend on the project.
* `optix_enable` : Enable OptiX which may boost performance, may be incompatible depending on the version of blender, project and GPU allocated

After you are done, go to Runtime > Run All (Ctrl + F9) and upload your files or have Google Drive authorised below. See the [GitHub repo](https://github.com/syn73/blender-colab) for more information.

In [19]:
blender_version = '5.0.0' #@param ['2.79b', '2.83.20', '2.93.18', '3.3.21', '3.6.23', '4.2.20', '4.5.9','5.0.0', '5.1.1'] {allow-input: true}
blend_file_path = 'path/to/file.blend' #@param {type: 'string'}
#@markdown ---
upload_type = 'google_drive' #@param ['direct', 'google_drive', 'url', 'gdrive_relative'] {allow-input: false}
drive_path = 'M_Bedroom_colab.blend' #@param {type: 'string'}
url_blend = 'https://drive.google.com/file/d/1yCtczsAxt5833Yi8R9aZbEi0LQ09mf1S/view?usp=drive_link' #@param {type: 'string'}
#@markdown ---
animation = True #@param {type: 'boolean'}
start_frame =  1#@param {type: 'integer'}
end_frame =  215#@param {type: 'integer'}
#@markdown ---
download_type = 'google_drive' #@param ['direct', 'google_drive', 'gdrive_direct'] {allow-input: false}
output_name = 'blender-##' #@param {type: 'string'}
zip_files = True #@param {type: 'boolean'}
drive_output_path = 'colab/output' #@param {type: 'string'}
#@markdown ---
gpu_enabled = True #@param {type:"boolean"}
optix_enabled = True #@param {type:"boolean"}
cpu_enabled = False #@param {type:"boolean"}

In [20]:
%cd /content

gpu = !nvidia-smi --query-gpu=gpu_name --format=csv,noheader
print("Current GPU: " + gpu[0])

if gpu[0] == "Tesla K80" and optix_enabled:
  print("OptiX disabled because of unsupported GPU")
  optix_enabled = False

/content
Current GPU: Tesla T4


In [21]:
import os

os.environ["LD_PRELOAD"] = ""

!apt remove libtcmalloc-minimal4
!apt install libtcmalloc-minimal4

os.environ["LD_PRELOAD"] = "/usr/lib/x86_64-linux-gnu/libtcmalloc_minimal.so.4.5.9"

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following packages will be REMOVED:
  libtcmalloc-minimal4
0 upgraded, 0 newly installed, 1 to remove and 51 not upgraded.
After this operation, 382 kB disk space will be freed.
(Reading database ... 122394 files and directories currently installed.)
Removing libtcmalloc-minimal4:amd64 (2.9.1-0ubuntu3) ...
Processing triggers for libc-bin (2.35-0ubuntu3.8) ...
/sbin/ldconfig.real: /usr/local/lib/libhwloc.so.15 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbb.so.12 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libur_adapter_level_zero.so.0 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libumf.so.1 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtcm_debug.so.1 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbbmalloc_proxy.so.2 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbbmalloc.so.2 is not a s

In [22]:
import shutil
from google.colab import files, drive
uploaded_filename = ""

if upload_type == 'google_drive' or upload_type == 'gdrive_relative' or download_type == 'google_drive' or download_type == 'gdrive_direct':
    drive.mount('/drive')

if upload_type == 'direct':
    uploaded = files.upload()
    for fn in uploaded.keys():
        uploaded_filename = fn
elif upload_type == 'url':
    !wget -nc $url_blend
    uploaded_filename = os.path.basename(url_blend)
elif upload_type == 'google_drive':
    shutil.copy('/drive/MyDrive/' + drive_path, '.')
    uploaded_filename = os.path.basename(drive_path)

Drive already mounted at /drive; to attempt to forcibly remount, call drive.mount("/drive", force_remount=True).


In [23]:
!rm -r render
!mkdir render

if upload_type == 'gdrive_relative':
    if not drive_path.endswith('/'):
        drive_path += '/'
    !cp -r '/drive/MyDrive/{drive_path}.' 'render/'
elif uploaded_filename.lower().endswith('.zip'):
    !unzip -o $uploaded_filename -d 'render/'
elif uploaded_filename.lower().endswith('.blend'):
    shutil.copy(uploaded_filename, 'render/')
    blend_file_path = uploaded_filename
else:
    raise SystemExit("Invalid file extension, only .blend and .zip can be uploaded.")

rm: cannot remove 'render': No such file or directory


In [24]:
import requests
blender_url_dict = {
    '2.79b'   : "https://ftp.nluug.nl/pub/graphics/blender/release/Blender2.79/blender-2.79b-linux-glibc219-x86_64.tar.bz2",
    '2.83.20' : "https://ftp.nluug.nl/pub/graphics/blender/release/Blender2.83/blender-2.83.20-linux-x64.tar.xz",
    '5.0.0'   : "https://ftp.nluug.nl/pub/graphics/blender/release/Blender5.0/blender-5.0.0-linux-x64.tar.xz"
    # example
    # The link will be inferred automatically even though the version is not defined here, you may override this behaviour by defining a new version here
    # Add any custom Linux binaries, refer to https://ftp.nluug.nl/pub/graphics/blender/release or other sites that provide direct link download
}

if blender_version in blender_url_dict:
    blender_url = blender_url_dict[blender_version]
else:
    major_minor = ".".join(blender_version.split('.')[:2])
    blender_url = f"https://ftp.nluug.nl/pub/graphics/blender/release/Blender{major_minor}/blender-{blender_version}-linux-x64.tar.xz"

try:
    response = requests.head(blender_url, allow_redirects=True, timeout=10)
    if response.status_code != 200:
        print(f"Download failed for version '{blender_version}'.")
        print("Error downloading: You may need to define the download archive manually above.")
    else:
        base_url = os.path.basename(blender_url)
        print(f"Download URL: {blender_url}")
        print(f"Base filename: {base_url}")
except Exception as e:
    print(f"Error checking URL: {e}")
    print("Error downloading: You may need to define the download archive manually above.")

!mkdir $blender_version
!wget -nc $blender_url
!tar -xkf $base_url -C ./$blender_version --strip-components=1

Download URL: https://ftp.nluug.nl/pub/graphics/blender/release/Blender5.0/blender-5.0.0-linux-x64.tar.xz
Base filename: blender-5.0.0-linux-x64.tar.xz
--2026-05-13 14:20:29--  https://ftp.nluug.nl/pub/graphics/blender/release/Blender5.0/blender-5.0.0-linux-x64.tar.xz
Resolving ftp.nluug.nl (ftp.nluug.nl)... 145.220.21.40, 2001:67c:6ec:221:145:220:21:40
Connecting to ftp.nluug.nl (ftp.nluug.nl)|145.220.21.40|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 387739652 (370M) [application/octet-stream]
Saving to: ‘blender-5.0.0-linux-x64.tar.xz’

blender-5.0.0-linux 100%[===================>] 369.78M  13.4MB/s    in 36s     

2026-05-13 14:21:07 (10.4 MB/s) - ‘blender-5.0.0-linux-x64.tar.xz’ saved [387739652/387739652]



In [25]:
# Enable GPU rendering (or add custom properties here)
data = "import re\n"+\
    "import bpy\n"+\
    "scene = bpy.context.scene\n"+\
    "scene.cycles.device = 'GPU'\n"+\
    "prefs = bpy.context.preferences\n"+\
    "prefs.addons['cycles'].preferences.get_devices()\n"+\
    "cprefs = prefs.addons['cycles'].preferences\n"+\
    "print(cprefs)\n"+\
    "for compute_device_type in ('CUDA', 'OPENCL', 'NONE'):\n"+\
    "    try:\n"+\
    "        cprefs.compute_device_type = compute_device_type\n"+\
    "        print('Device found:',compute_device_type)\n"+\
    "        break\n"+\
    "    except TypeError:\n"+\
    "        pass\n"+\
    "for device in cprefs.devices:\n"+\
    "    if not re.match('intel', device.name, re.I):\n"+\
    "        print('Activating',device)\n"+\
    "        device.use = "+str(gpu_enabled)+"\n"+\
    "    else:\n"+\
    "        device.use = "+str(cpu_enabled)+"\n"
with open('setgpu.py', 'w') as f:
    f.write(data)

renderer = "CUDA"
if optix_enabled:
    print("Note: You're currently using OptiX renderer. If an error occurred, the current GPU (e.g. Tesla K80) is not supported and you need to switch back to CUDA.")
    renderer = "OPTIX"

Note: You're currently using OptiX renderer. If an error occurred, the current GPU (e.g. Tesla K80) is not supported and you need to switch back to CUDA.


In [26]:
%cd /content

!rm -r output
!mkdir output

if not drive_output_path.endswith('/'):
    drive_output_path += '/'

if download_type != 'gdrive_direct':
    output_path = '/content/output/' + output_name
else:
    output_path = '/drive/MyDrive/' + drive_output_path + output_name

%cd /content/$blender_version

if animation:
    if start_frame == end_frame:
        !./$blender_version/blender -b '/content/render/{blend_file_path}' -P "/content/setgpu.py" -E CYCLES -o '{output_path}' -noaudio -a -- --cycles-device "{renderer}"
    else:
        !./$blender_version/blender -b '/content/render/{blend_file_path}' -P "/content/setgpu.py" -E CYCLES -o '{output_path}' -noaudio -s $start_frame -e $end_frame -a -- --cycles-device "{renderer}"
else:
    !./$blender_version/blender -b '/content/render/{blend_file_path}' -P "/content/setgpu.py" -E CYCLES -o '{output_path}' -noaudio -f $start_frame -- --cycles-device "{renderer}"

Streaming output truncated to the last 5000 lines.
02:45:37.634  render           | Mem: 1M | Synchronizing object | zip.006
02:45:37.635  render           | Mem: 1M | Synchronizing object | zip.002
02:45:37.635  render           | Mem: 1M | Synchronizing object | keychain.2
02:45:37.635  render           | Mem: 1M | Synchronizing object | hooks_002.001
02:45:37.641  render           | Mem: 1M | Synchronizing object | interio_002.002
02:45:37.676  render           | Mem: 1M | Synchronizing object | wheel_base_001
02:45:37.692  render           | Mem: 1M | Synchronizing object | M_spinner_wheel.004
02:45:37.704  render           | Mem: 1M | Synchronizing object | Hinge.005
02:45:37.780  render           | Mem: 1M | Synchronizing object | TSA_LOCK.003
02:45:37.803  render           | Mem: 1M | Synchronizing object | TSA_LOCK.004
02:45:37.820  render           | Mem: 1M | Synchronizing object | TSA_LOCK.005
02:45:37.830  render           | Mem: 1M | Synchronizing object | LOCK_NO.001
02:4

In [28]:
import os
import shutil
from google.colab import files

%cd /content

# SETTINGS
download_type = 'gdrive'   # 'direct' or 'gdrive'
drive_output_path = 'colab/output/'   # Google Drive folder
zip_files = False
output_name = "render_"

# FIND OUTPUT FILES
path, dirs, files_folder = next(os.walk("output"))

# FILTER ONLY EXR FILES
files_folder = [f for f in files_folder if f.lower().endswith('.exr')]

# OUTPUT ZIP NAME
output_folder_name = output_name.replace('#', '') + 'render'

# CREATE DRIVE FOLDER IF NEEDED
drive_dst = '/drive/MyDrive/' + drive_output_path
os.makedirs(drive_dst, exist_ok=True)

# NO FILES
if len(files_folder) == 0:
    raise SystemExit("No EXR frames were rendered.")

# SINGLE FILE
elif len(files_folder) == 1:

    render_img = 'output/' + files_folder[0]

    if download_type == 'direct':

        files.download(render_img)

    else:

        src = '/content/' + render_img
        dst = os.path.join(drive_dst, files_folder[0])

        shutil.copy(src, dst)

        print(f"Copied to: {dst}")

# MULTIPLE FILES
elif len(files_folder) > 1:

    # ZIP MODE
    if zip_files:

        shutil.make_archive(output_folder_name, 'zip', 'output')

        if download_type == 'direct':

            files.download(output_folder_name + '.zip')

        else:

            src = '/content/' + output_folder_name + '.zip'
            dst = os.path.join(drive_dst, output_folder_name + '.zip')

            shutil.copy(src, dst)

            print(f"ZIP copied to: {dst}")

    # NORMAL COPY MODE
    else:

        for f in files_folder:

            src = "/content/output/" + f
            dst = os.path.join(drive_dst, f)

            shutil.copy(src, dst)

            print(f"Copied: {f}")

print("DONE")

/content
Copied: blender-04.exr
Copied: blender-116.exr
Copied: blender-35.exr
Copied: blender-34.exr
Copied: blender-121.exr
Copied: blender-03.exr
Copied: blender-55.exr
Copied: blender-152.exr
Copied: blender-101.exr
Copied: blender-196.exr
Copied: blender-155.exr
Copied: blender-23.exr
Copied: blender-28.exr
Copied: blender-74.exr
Copied: blender-99.exr
Copied: blender-61.exr
Copied: blender-129.exr
Copied: blender-153.exr
Copied: blender-86.exr
Copied: blender-113.exr
Copied: blender-22.exr
Copied: blender-182.exr
Copied: blender-102.exr
Copied: blender-181.exr
Copied: blender-15.exr
Copied: blender-148.exr
Copied: blender-160.exr
Copied: blender-171.exr
Copied: blender-01.exr
Copied: blender-45.exr
Copied: blender-184.exr
Copied: blender-44.exr
Copied: blender-48.exr
Copied: blender-30.exr
Copied: blender-92.exr
Copied: blender-02.exr
Copied: blender-93.exr
Copied: blender-91.exr
Copied: blender-208.exr
Copied: blender-185.exr
Copied: blender-178.exr
Copied: blender-186.exr
Copie

## Disclaimer
Google Colab is targeted to researchers and students to run AI/ML tasks, data analysis and education, not rendering 3D scenes. Because the computing power provided are free, the usage limits, idle timeouts and speed of the rendering may varies time by time. [Colab Pro and Colab Pro+](https://colab.research.google.com/signup) are available for those who wanted to have more powerful GPU and longer runtimes for rendering. See the [FAQ](https://research.google.com/colaboratory/faq.html) for more info. In some cases, it might be faster to use an online Blender renderfarm.

## License
```
MIT License

Copyright (c) 2020-2022 ynshung

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
SOFTWARE.
```